In [ ]:
!wget https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak/master/python/sylbreak.py

--2025-12-23 13:19:46--  https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak/master/python/sylbreak.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3668 (3.6K) [text/plain]
Saving to: ‘sylbreak.py’

sylbreak.py         100%[===================>]   3.58K  --.-KB/s    in 0s      

2025-12-23 13:19:46 (62.5 MB/s) - ‘sylbreak.py’ saved [3668/3668]



In [ ]:
!pip install transformers==4.41.1 peft==0.11.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 58.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
  Attempting uninstall: peft
    Found existing installation: peft 0.18.0
    Uninstalling peft-0.18.0:
      Successfully uninstalled peft-0.18.0


In [ ]:
!pip install 'datasets[audio]==2.14.4' 'fsspec==2023.9.2'

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.3/519.3 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.4/173.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:
      Successfully uninstalled multiprocess-0.70.16
  Attempting uninstall: datasets
    Found

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
speech_data = load_dataset("LULab/mediTalk-mm-rdy", split='test')

In [ ]:
from sylbreak import break_syllables, create_break_pattern

def syllable_break(text):
  """Syllable break for burmese texts"""
  text = text
  separator = ' '
  break_pattern = create_break_pattern()

  segmented = break_syllables(text, break_pattern, separator)
  return segmented

In [ ]:
def apply_syllable_break(text):
    text['prompt'] = syllable_break(text['prompt'])
    return text

dataset = speech_data.map(apply_syllable_break)
dataset

Dataset({
    features: ['speaker_id', 'prompt', 'audio'],
    num_rows: 2920
})

In [ ]:
from transformers import pipeline
import torch

MODEL_NAME = "openai/whisper-large-v3"  # specify the model name
lang = "my"  # change to Thai langauge

device = 0 if torch.cuda.is_available() else "cpu"

pipe = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_NAME,
    chunk_length_s=10,
    stride_length_s=2,
    device=device,
)
pipe.model.config.forced_decoder_ids = pipe.tokenizer.get_decoder_prompt_ids(
  language=lang,
  task="transcribe"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

In [ ]:
sample_dataset = dataset.select(range(100))
sample_dataset

Dataset({
    features: ['speaker_id', 'prompt', 'audio'],
    num_rows: 100
})

In [ ]:
sample_dataset['prompt'][0]

'ဆေး ရုံ က ဈာ ပ န ကို စီ စဉ် နေ တယ် ဆို ရင် ကျွန် တော် တို့ က အုတ် ဂူ ကို အ မှတ် အ သား မ ပြု လုပ် တဲ့ အ တွက် ခင် ဗျား က လေး ကို ဘယ် နေ ရာ မှာ မြှပ် လိုက် လဲ သိ မှာ မ ဟုတ် လို့ ကျေး ဇူး ပြု ပြီး အ သု ဘ ကို တက် ရောက် ပေး ပါ'

In [ ]:
from tqdm import tqdm
import time
import torch

total_audio_time = 0.0
total_inference_time = 0.0

pipe(speech_data[0]["audio"])

for item in tqdm(sample_dataset):

    audio = item["audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    total_audio_time += duration

    torch.cuda.synchronize()
    start = time.time()

    _ = pipe(audio)

    torch.cuda.synchronize()
    end = time.time()

    total_inference_time += (end - start)

rtf = total_inference_time / total_audio_time

print()
print()

print(f"RTF: {rtf:.3f}")


100%|██████████| 100/100 [16:53<00:00, 10.14s/it]



RTF: 1.696


In [ ]:
dataset = load_dataset("LULab/mediTalk-mm")
dataset

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


DatasetDict({
    train: Dataset({
        features: ['filename', 'speaker_id', 'label', 'prompt', 'audio_array'],
        num_rows: 26264
    })
    test: Dataset({
        features: ['filename', 'speaker_id', 'label', 'prompt', 'audio_array'],
        num_rows: 2920
    })
})

In [ ]:
unique_speakers = dataset["train"].unique("speaker_id")
print(len(unique_speakers))
print(unique_speakers)


9
['speaker202504', 'speaker202501', 'speaker202506', 'speaker202507', 'speaker202503', 'speaker202502', 'speaker202508', 'speaker202505', 'speaker202509']


In [ ]:
RTF (FFT Medium): 0.691
RTF (FFT Base): 0.204
RTF (FFT Tiny): 0.148

RTF (FFT Tiny Augmentation): 0.144
RTF (FFT Medium Augmentation): 0.696

RTF (PEFT Large v2): 7.846
RTF (PEFT Small): 2.874
